Data quality makes sure records meet expectations. Data validation checks business and technical rules. Quarantine isolates invalid records for investigation. Deduplication selects the latest valid version. Error handling detects and reports pipeline failures. Failure recovery diagnoses and safely reruns processes. Finally, data lineage and audit traces records and changes


flowchart TD
    A["🛢️ Incoming order records<br/>Valid rows, missing IDs, negative amounts,<br/>invalid amounts, duplicates"]

    B["Data quality validation<br/>Check required fields, data types,<br/>and business rules"]

    C["✅ Valid records<br/>Proceed to Silver transformations"]

    D["⚠️ Quarantine<br/>Retain invalid rows and rejection reasons"]

    E["🔄 Recovery and monitoring<br/>Diagnose failures, correct data,<br/>and safely retry"]

    A --> B
    B --> C
    B --> D
    C --> E
    D --> E

    classDef default fill:#181818,stroke:#444,color:#ffffff
    classDef valid fill:#181818,stroke:#444,color:#ffffff
    classDef invalid fill:#181818,stroke:#444,color:#ffffff

    class A,B,E default
    class C valid
    class D invalid


functions as F concept is PySpark column expressions. 
Window is latest record deduplication. 
StructType is explicit source schema. 
valid_table is destination for quality-approved records
quarantine_table is destination for invalid records
current_table is destination for deduplicated current orders
Why separate tables? We want to preserve the rejected data without contaminating our trusted output

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType

catalog = "workspace"
schema = "incremental_project"

valid_table = f"{catalog}.{schema}.orders_valid_step6"
quarantine_table = f"{catalog}.{schema}.orders_quarantine_step6"
current_table = f"{catalog}.{schema}.orders_current_step6"

print("Valid table:", valid_table)
print("Quarantine table:", quarantine_table)
print("Current-state table:", current_table)

Controlled test data is implemented at spark.createDataFrame(sample_data, source_schema). This is useful because in production, we should test that pipelines behave correctly for invalid inputs, not only for perfectly clean data
Nullable source fields is implemented at StructField("order_id", StringType(), True). True allows null values at source level so we can detect and handle them during validation

In [ ]:
source_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_amount", StringType(), True),
    StructField("updated_at", StringType(), True)
])

sample_data = [
    ("ORD020", "C120", "delivered", "250.00",
     "2026-10-04 10:00:00"),

    ("ORD021", "C121", "shipped", "-50.00",
     "2026-10-04 10:05:00"),

    ("ORD022", "C122", "processing", "invalid",
     "2026-10-04 10:10:00"),

    (None, "C123", "shipped", "125.00",
     "2026-10-04 10:15:00"),

    ("ORD023", "C124", "processing", "100.00",
     "2026-10-04 10:20:00"),

    ("ORD023", "C124", "shipped", "100.00",
     "2026-10-05 10:20:00")
]

raw_df = spark.createDataFrame(sample_data, source_schema)

display(raw_df)

In [ ]:
assert raw_df.count() == 6
print("Source data created successfully")

Data quality rules that follow four validations:
1. order_id must not be missing
2. order_amount must be a valid decimal
3. order_amount cannot be negative
4. updated_at must contain a valid timestamp

Safe type conversion is implemented at F.expr("try_cast(order_amount as DECIMAL(12,2))") with try_cast attempting to convert the string into a decimal. Unlike strict casting that can throw an error for malformed input, try_cast returns NULL when a supported conversion falls. This is useful when we need to inspect and quarantine bad data.
Same principle applies to timestamps with F.expr("try_cast(updated_at AS TIMESTAMP)")

In [ ]:
validated_df = (
    raw_df
    .withColumn(
        "amount_decimal",
        F.expr("try_cast(order_amount AS DECIMAL(12,2))")
    )
    .withColumn(
        "updated_timestamp",
        F.expr("try_cast(updated_at AS TIMESTAMP)")
    )
)

Conditional validation is implemented at F.when(condition, "REJECTION_REASON") and F.when() works like a SQL CASE WHEN. It evaluates conditions and assigns a value when a condition is satisfied. For example: F.when(F.col("amount_decimal")<0, "NEGATIVE_AMOUNT") identiefies records containing a negative order amount. Data quality classification is implemented at .withColumn("rejection_reason",...). Rather than simply dropping bad records, we classify them. This makes it easier for engineer to investigate the cause of a pipeline problem.

In [ ]:
validated_df = (
    validated_df
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("order_id").isNull() |
            (F.trim(F.col("order_id")) == ""),
            "MISSING_ORDER_ID"
        )
        .when(
            F.col("amount_decimal").isNull(),
            "INVALID_AMOUNT"
        )
        .when(
            F.col("amount_decimal") < 0,
            "NEGATIVE_AMOUNT"
        )
        .when(
            F.col("updated_timestamp").isNull(),
            "INVALID_TIMESTAMP"
        )
        .otherwise(None)
    )
)

display(
    validated_df.select(
        "order_id",
        "order_amount",
        "amount_decimal",
        "rejection_reason"
    )
)

In [ ]:
display(
    validated_df.groupBy("rejection_reason").count()
)

Separating valid and invalid records
Valid records are implemented at F.col("rejection_reason").isNull() records without a rejection reason are allowed to continue. Quarantine is implemented at F.col("rejection_reason").isNotNull() which is records contianing a validation error that are separated for investigation. Bad records are not deleted, because quarantining preserves the original record and reason it was rejected. 

In [ ]:
valid_df = validated_df.filter(
    F.col("rejection_reason").isNull()
)

quarantine_df = validated_df.filter(
    F.col("rejection_reason").isNotNull()
)

print("Valid rows:", valid_df.count())
print("Quarantined rows:", quarantine_df.count())

.format("delta") is a transactional delta storage. .mode("overwrite") replace the contents of a practice table. .option("overwriteSchema","true") allow replacement of the stored schema. .saveAsTable(...) persist the DataFrame as a named table
In a production pipeline, we wouldn't routinely overwrite quarantine history. We'd use append or idempotent MERGE logic, along with batch identifiers
assert accepted + rejected == raw_df.count() checks that every incoming record is accounted for as either accepted or rejected

In [ ]:
(
    valid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(valid_table)
)

(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(quarantine_table)
)

assert accepted + rejected == raw_df.count() checks that every incoming record is accounted for as either accepted or rejected which is concept of reconciliation

In [ ]:
accepted = spark.table(valid_table).count()
rejected = spark.table(quarantine_table).count()

assert accepted == 3
assert rejected == 3
assert accepted + rejected == raw_df.count()

print("Record reconciliation passed!")

Window.paritionBy("order_id") groups order versions and puts all versions of each order in the same logical window group
F.col("updated_timestamp").desc() sorts newer versions first
F.row_number().over(order_window) assigns a row number to each order version
.filter(F.col("row_num")==1) keeps the latest valid version per order



In [ ]:
order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("updated_timestamp").desc()
    )
)

ranked_valid = valid_df.withColumn(
    "row_num",
    F.row_number().over(order_window)
)

current_orders = (
    ranked_valid
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

display(
    current_orders.select(
        "order_id",
        "order_status",
        "amount_decimal",
        "updated_timestamp"
    ).orderBy("order_id")
)

In [ ]:
assert current_orders.count() == 2

assert(
    current_orders.select("order_id").distinct().count()
    == 2
)

display(
    current_orders.select(
        "order_id","order_status"
        )
)

print("Latest valid order selection passed!")

assert rejected_count == 0 specifies an acceptance requirement which is quality gate
raise

assert rejected_count == 0 specifies an acceptance requirement which quality gate. raise through failed assert signals that a requirement failed which is exception. try runs code that might fail which is the error handling. except AssertionError captures the expected failure for inspection that is exception handling.

In a production job, a critical quality failure should normally cause task to fail and alert responsible team. With bad data, the important principle is to make failure policy explicit and not to silently ignore invalid data

In [ ]:
try:
    rejected_count = quarantine_df.count()

    assert rejected_count == 0, (
        f"Data quality gate failed: "
        f"{rejected_count} invalid records found"
    )

    print("Quality gate passed")

except AssertionError as error:
    print("Quality gate blocked processing:")
    print(error)

Quarantine three invalid records, retain three accepted records, and publish two latest valid orders into a separate practice table
current_orders.write.format("delta") publish only records that passed our validation and latest-record selection
.mode("overwrite") allows rerunning this write replaces its current contents rather than appending duplicates. 
We have recovered the valid processing path but we have not corrected the invalid source records. They remain in quarantine for later remediation
In a real incident, recovery might include fixing upstream source, replaying corrected records, checking for missing data, and verifying downstream results

In [ ]:
(
    current_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(current_table)
)

recovered_df = spark.table(current_table)

assert recovered_df.count() == 2

assert (
    recovered_df.select("order_id").distinct().count()
    == 2
)

print("Valid records published successfully")
print("Invalid source records remain quarantine")

In [ ]:
display(
    spark.table(quarantine_table)
    .select(
        "order_id",
        "order_amount",
        "rejection_reason"
    )
)

DESCRIBE HISTORY table_name can show operations such as WRITE, UPDATE, MERGE, and other modifications, depending on what actually happened to the table

Helps answer questions like: When was this table last modified? What type of operation occurred?

Delta Lake also supports time travel and restoration to earlier versions, provided necessary history and data files are still retained. These capabilities have retention and permission limitations, so don't treat table history as a permanent backup

In [ ]:
spark.sql(
    f"DESCRIBE HISTORY {current_table}"
).select(
    "version",
    "timestamp",
    "operation"
).show(truncate=False)

Structured Streaming stores progress info needed to resume a stream after failures. Changing or deleting checkpoint can cause previously processed files to be treated as new work

We use try_cast to safely convert incoming values. When conversion fails, it returns NULL, which we can detect and quarantine instead of allowing a malformed record to fail processing job

A quarantine table stores records that fail data quality checks, along with their rejection reasons. This preserves traceability and allows engineers to investigate and reprocess corrected data

Reconciliation verifies that all incoming records are accounted for between accepted and rejected datasets. It helps detect data loss or unexpected duplication during processing

Exception handling detects and manages failures during execution. Data recovery goes further by correcting underlying issue, safely reprocessing affected data, and validating final state. Actual recovery looks like 1. Detecting the problem 2. Quarantining the record 3. Investigating and correcting 4. Reprocessing 5. Reconciliation. Exception handling detects and manages failures during execution. Data recovery goes further by correcting underlying issue, safety reprocessing affected data, and validating final state.

A streaming checkpoint stores processing state so a streaming job can resume after failure. Delta table history records committed table versions and operations, which helps with auditing, troubleshooting, and examining historical changes.

